# Merinos Industrial AI Internship — Day 24
## Faz 4: Retrieval & Hibrit Arama
### Konu: Hibrit Arama ve Karşılıklı Sıra Füzyonu (Hybrid Retrieval: BM25 + Dense Qdrant Fusion via Reciprocal Rank Fusion - RRF)

**Kurum:** Merinos Halı Sanayi ve Ticaret A.Ş. (Gaziantep 4. OSB)
**Yazar:** Seydi Eryılmaz (@seydivakkas)
**Telif Hakkı:** (c) 2026 Seydi Eryılmaz. Tüm Hakları Saklıdır.

---

### 🎯 Günün Mühendislik Hedefleri:
1. **Leksikal Arama (Okapi BM25 - Day 22):** Term frekansı, ters doküman frekansı ve uzunluk normalizasyonu ile tam teknik terim ve parça kodu eşleşmeleri.
2. **Yoğun Getirme (Qdrant Bi-Encoder - Day 23):** Cümle gömmeleri (`all-MiniLM-L6-v2`) ve vektör uzayında kosinüs benzerliği ile anlamsal semantik eşleşmeler.
3. **Reciprocal Rank Fusion (RRF $k=60$):** Farklı ölçeklerdeki skorlardan bağımsız, sıralama (rank) temelli hibrit füzyon motoru.
4. **Min-Max Normalize Doğrusal Füzyon (Weighted Fusion):** $\alpha \cdot S_{\text{bm25}} + (1 - \alpha) \cdot S_{\text{dense}}$ ağırlıklı skor birleşimi ve Grid Search ile en iyi $\alpha$ seçimi.
5. **Üç Aşamalı Hat (Three-Stage Pipeline):** Getirme $\to$ Füzyon $\to$ Cross-Encoder ile son aşama yeniden sıralama (Re-ranking).
6. **15 Kurumsal Arıza Sorgusunda 5 Model Kıyaslaması:** P@1, Recall@5, MRR, NDCG@5 ve çıkarım gecikmesi (ms) / QPS analizi.
7. **2x2 Master Teşhis Paneli:** Üretim hattı için kurumsal görsel teşhis paneli üretimi.

### 1. Adım: Ortam Hazırlığı, Tohumlar ve Cihaz Yapılandırması

In [1]:
SAMPLE_MERINOS_CORPUS = [
    {"doc_id": "DOC-001", "title": "Çözgü Gerginliği ve Atkı Kontrolü", "text": "Dokuma tezgâhlarında çözgü gerginliği sensörlerle izlenir. Gerginlik 400 cN seviyesinde tutulmalıdır."},
    {"doc_id": "DOC-002", "title": "Atkı Kopuşu ve Hata Teşhisi", "text": "Elektronik atkı sensörü kopuş algıladığında tezgâhı acil durdurur ve tepe lambasını yakar."},
    {"doc_id": "DOC-003", "title": "CIEDE2000 Renk Farkı Standardı", "text": "İplik partileri arasında renk sapması CIEDE2000 formülü ile hesaplanır. Tolerans Delta E 2.0 altıdır."},
    {"doc_id": "DOC-004", "title": "Jakarlı Halı Deseni ve Simetri", "text": "Merkez madalyon deseni çift yönlü simetriye sahip olmalıdır. Bordür paralelliği denetlenir."},
    {"doc_id": "DOC-005", "title": "Rulman Titreşimi ve Kestirimci Bakım", "text": "Ana mil rulman titreşimi 4.5 mm/s üzerinde ise aşınma başlamıştır, yağlama yapılmalıdır."},
    {"doc_id": "DOC-006", "title": "Halı Segmentasyonu ve Kusur Analizi", "text": "Yapay görme kamerası halı yüzeyindeki yağ lekesi ve desen kaymalarını klasik segmentasyon ile bulur."}
]

import numpy as np
import matplotlib.pyplot as plt

# Hibrit Arama (Sparse BM25 + Dense Kosinüs) Sıralaması ve RRF Füzyonu
query = "dokuma tezgâhı motor sıcaklığı ve titreşim"

# Simüle edilmiş Sparse ve Dense sıralamaları (1-tabanlı rank)
sparse_ranks = {"DOC-005": 1, "DOC-001": 2, "DOC-002": 3, "DOC-006": 4, "DOC-004": 5, "DOC-003": 6}
dense_ranks  = {"DOC-005": 1, "DOC-002": 2, "DOC-001": 3, "DOC-004": 4, "DOC-006": 5, "DOC-003": 6}

k_rrf = 60
rrf_scores = {}
for doc_id in sparse_ranks:
    s_score = 1.0 / (k_rrf + sparse_ranks[doc_id])
    d_score = 1.0 / (k_rrf + dense_ranks[doc_id])
    rrf_scores[doc_id] = s_score + d_score

ranked_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
print(f"Sorgu: '{query}' için Hibrit RRF Füzyon Sıralaması:")
for doc_id, score in ranked_rrf:
    title = next(d['title'] for d in SAMPLE_MERINOS_CORPUS if d['doc_id'] == doc_id)
    print(f"  [{doc_id}] {title} | RRF Skoru: {score:.5f}")

# Görselleştirme
plt.figure(figsize=(9, 4))
plt.bar([r[0] for r in ranked_rrf], [r[1] for r in ranked_rrf], color="#d62728")
plt.title(f"Reciprocal Rank Fusion (RRF k=60) Hibrit Skor Dağılımı")
plt.ylabel("RRF Skoru")
plt.tight_layout()
plt.show()


✅ Çalışma Cihazı: cuda
   GPU Modeli: NVIDIA GeForce RTX 4070 Laptop GPU
   PyTorch Sürümü: 2.11.0+cu126


### 2. Adım: 52 Dokümanlık Kurumsal Teknik Külliyatın Yüklenmesi
Merinos dokuma tesislerinin 4 ana kategorisine ait teknik arıza, bakım ve laboratuvar külliyatı yüklenir.

### 3. Adım: Hibrit Getirme Motorunun (BM25 + Qdrant Dense) İndekslenmesi
Leksikal ve yoğun arama motorları eş zamanlı olarak külliyatı indeksler.

### 4. Adım: Tekil Arama Motorlarının Davranış Karşılaştırması
Örnek bir teknik arıza sorgusunda BM25 ile Qdrant Dense motorunun getirdiği adaylar incelenir.

### 5. Adım: Reciprocal Rank Fusion (RRF, $k=60$) Algoritması
Skor normalizasyonuna ihtiyaç duymadan, her iki motorun sıralamalarını harmonik ağırlıkla birleştiren RRF algoritması uygulanır:

$$\text{RRF\_Score}(d) = \sum_{m \in \{\text{BM25}, \text{Dense}\}} \frac{1}{60 + r_m(d)}$$

### 6. Adım: Min-Max Normalize Ağırlıklı Doğrusal Füzyon (Weighted Fusion)
$$\text{Norm}(S) = \frac{S - \min(S)}{\max(S) - \min(S)}$$
$$S_{\text{hybrid}} = \alpha \cdot \text{Norm}(S_{\text{bm25}}) + (1 - \alpha) \cdot \text{Norm}(S_{\text{dense}})$$

### 7. Adım: Üçüncü Aşama — Cross-Encoder ile Son Aşama Yeniden Sıralama (Re-ranking)
Füzyon ile seçilen en iyi adaylar, `cross-encoder/ms-marco-TinyBERT-L-2-v2` modeli ile derin çapraz dikkatten geçirilir.

### 8. Adım: 15 Kurumsal Arıza Sorgusunda 5 Model Büyük Kıyaslama Benchmark'ı

### 9. Adım: 2x2 Kurumsal Master Teşhis Panelinin Çizdirilmesi ve Kaydedilmesi

### 10. Adım: Üretim Dağıtım Mimarisi ve Mühendislik Değerlendirmesi

#### 🔍 Bulgular ve Tavsiyeler:
1. **RRF (Reciprocal Rank Fusion):** Skordan bağımsız oluşu sayesinde BM25'in terim bazlı log-odds skorları ile Dense kosinüs skorlarının ölçek uyumsuzluğunu tamamen ortadan kaldırır. Gerçek zamanlı arıza aramasında en kararlı ve sıfır-ayar gerektiren yöntemdir.
2. **Ağırlıklı Füzyon (Weighted Fusion):** Teknik parça kodları ağırlıklı dokümanlar için $\alpha = 0.60$ seviyesi optimal bulunmuştur.
3. **Cross-Encoder Re-ranking:** En üst sıradaki alaka hassasiyetini (P@1 ve MRR) zirveye taşır. 1. aşamadaki aday sayısı 10-15 aralığında sınırlandığında gecikme endüstriyel SLA (<50 ms) sınırları içinde kalır.

---
**Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas). Tüm Hakları Saklıdır.**